# Prepare shared nonlinear Lorenz-96 data

This notebook creates or verifies the frozen `lorenz96` bundles used by
all seven method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same declared ground truth.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
import tempfile
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
def series_digest(series):
    return hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()


def save_bundle(path, *, series, aggregate_truth, metadata):
    path.parent.mkdir(parents=True, exist_ok=True)
    digest = series_digest(series)
    metadata = {**metadata, "data_digest": digest}
    metadata_path = path.with_suffix(".json")
    if path.exists() or metadata_path.exists():
        if not path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Incomplete dataset bundle at {path}.")
        recorded = json.loads(metadata_path.read_text(encoding="utf-8"))
        with np.load(path, allow_pickle=False) as stored:
            saved_series = np.asarray(stored["series"], dtype=float)
        if series_digest(saved_series) != digest or recorded != metadata:
            raise ValueError(f"Existing bundle does not match request: {path}.")
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as handle:
        np.savez_compressed(
            handle,
            series=np.asarray(series, dtype=float),
            aggregate_truth=np.asarray(aggregate_truth, dtype=bool),
        )
        temporary_data = Path(handle.name)
    os.replace(temporary_data, path)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{metadata_path.name}.",
        delete=False,
    ) as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")
        temporary_metadata = Path(handle.name)
    os.replace(temporary_metadata, metadata_path)
    return True

In [ ]:
DATASET_KEY = "lorenz96"
DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
UNN_ROOT = Path(
    os.environ.get("UNN_ROOT", PROJECT_ROOT / "external" / "UNN")
).expanduser()
UPSTREAM_REVISION = verify_upstream_revision(
    UNN_ROOT,
    expected_url="https://github.com/jarrycyx/UNN.git",
    expected_commit="2212724cce8a7752db564089a7537fecedb0e36d",
)
CUTS_ROOT = UNN_ROOT / "CUTS_Plus"
if str(CUTS_ROOT) not in sys.path:
    sys.path.insert(0, str(CUTS_ROOT))
from data.simu_data import simulate_lorenz_96

VARIABLES = (10, 20)
SAMPLES = (250, 500, 1000)
REPLICATES = 50
SEED_START = 20_000
FIT_ORDER = 5
MAX_NEW_DATASETS: int | None = None

created = 0
for n_variables in VARIABLES:
    for n_samples in SAMPLES:
        for seed in range(SEED_START, SEED_START + REPLICATES):
            if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
                break
            data, target_source_truth = simulate_lorenz_96(
                p=n_variables,
                T=n_samples,
                F=10.0,
                delta_t=0.1,
                sd=0.1,
                burn_in=1000,
                seed=seed,
            )
            dataset_id = f"lorenz96-var{n_variables}-t{n_samples}-seed{seed}"
            was_created = save_bundle(
                DATASET_DIR / f"{dataset_id}.npz",
                series=np.asarray(data, dtype=float).T,
                aggregate_truth=np.asarray(target_source_truth, dtype=bool).T,
                metadata={
                    "dataset_id": dataset_id,
                    "dataset": DATASET_KEY,
                    "generator": "UNN/CUTS_Plus/data/simu_data.py:simulate_lorenz_96",
                    "upstream_commit": UPSTREAM_REVISION.commit,
                    "seed": seed,
                    "n_variables": n_variables,
                    "n_samples": n_samples,
                    "fit_order": FIT_ORDER,
                    "series_layout": "variable, time",
                    "truth_layout": "source, target",
                    "truth_scope": "aggregate structural dependency",
                },
            )
            created += int(was_created)
        if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
            break
    if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
        break
print(f"Created {created} new Lorenz-96 bundles. Rerun to resume.")